# DiGToR on SemanticRT - end-to-end

Disagreement-Guided Token Routing for RGB-Thermal semantic segmentation, run on
the **SemanticRT** benchmark (12 foreground classes + background, 11,371 RGB-T
pairs, official train/val/test split). Same pipeline as the FMB notebook; only
the dataset layout differs (flat `rgb/ thermal/ labels/` folders + `*.txt`
split files instead of FMB's `train/ test/` partition).

Run the cells top to bottom on Colab (Pro / A100) or Kaggle; the notebook is
self-contained:

1. Clone the repo.
2. Download the SemanticRT dataset zip from Google Drive and unzip it.
3. Train the v_only / t_only teachers, the fusion baseline, and DiGToR.
4. Evaluate the detector, rescue protocol, robustness, FLOPs, and modality-cut.


In [ ]:
import os
import shutil

repo_name = "DiGToR"

# Neu thu muc da ton tai, xoa di de chuan bi tai moi
if os.path.exists(repo_name):
    print(f"Phat hien thu muc '{repo_name}' da ton tai. Dang xoa...")
    shutil.rmtree(repo_name)
    print("Da xoa thu muc cu.")

repo_url = f"https://github.com/nguyenmaiductrong/{repo_name}.git"

print(f"Dang tai repo tu {repo_url}...")
exit_code = os.system(f"git clone {repo_url}")
print("Tai thanh cong." if exit_code == 0 else "Co loi khi tai, kiem tra URL/mang.")


In [ ]:
# repo da duoc clone moi o cell tren (fresh = latest main); pull nay chi de chac chan
!cd DiGToR && git pull origin main || echo "(pull skipped - fresh clone already latest)"


In [ ]:
cd DiGToR


In [ ]:
# --- Download SemanticRT from Google Drive (single zip) and unzip the flat layout ---
# Data is stored OUTSIDE the cloned repo so a re-clone does not wipe it, and the
# step is idempotent: re-running skips the download if the data is already there.
import os, glob, zipfile, shutil, subprocess, sys

# Drive FILE id of the SemanticRT zip (from the share link .../file/d/<ID>/view).
DRIVE_FILE_ID = "1KXgqYLy-yQBLAzhXY1zusVQiQ27Gr0Uw"
SEMRT_DIR = "/content/SemanticRT_dataset" if os.path.isdir("/content") else \
    os.path.join(os.path.dirname(os.getcwd()), "SemanticRT_dataset")
_mods = ("rgb", "thermal", "labels")
_splits = ("train.txt", "val.txt", "test.txt")

def _ready(base):
    return (all(os.path.isdir(os.path.join(base, m)) for m in _mods)
            and all(os.path.isfile(os.path.join(base, s)) for s in _splits))

def _dataroot(base):
    # the directory that directly holds rgb/thermal/labels + the split txt files,
    # whatever the zip's internal nesting is
    for root, _, _ in os.walk(base):
        if _ready(root):
            return root
    raise FileNotFoundError(f"no rgb/thermal/labels(+txt) folder under {base}")

if _ready(SEMRT_DIR):
    print("SemanticRT already prepared at", SEMRT_DIR)
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
    import gdown
    dl = os.path.join(os.getcwd(), "_semrt_download")
    shutil.rmtree(dl, ignore_errors=True); os.makedirs(dl, exist_ok=True)
    zpath = os.path.join(dl, "semanticrt.zip")
    gdown.download(id=DRIVE_FILE_ID, output=zpath, quiet=False)
    tmp = os.path.join(os.getcwd(), "_semrt_unzip")
    shutil.rmtree(tmp, ignore_errors=True)
    print("unzipping semanticrt.zip ...")
    with zipfile.ZipFile(zpath) as z:
        z.extractall(tmp)
    shutil.rmtree(SEMRT_DIR, ignore_errors=True)
    shutil.move(_dataroot(tmp), SEMRT_DIR)
    shutil.rmtree(tmp, ignore_errors=True)
    shutil.rmtree(dl, ignore_errors=True)
    print("Prepared SemanticRT at", SEMRT_DIR)

# The detection cell below reads SEMRT_ROOT first, so this works regardless of
# where the data was staged.
os.environ["SEMRT_ROOT"] = SEMRT_DIR
for m in _mods:
    n = len(glob.glob(os.path.join(SEMRT_DIR, m, "*")))
    print(f"  {m}: {n} files")


In [ ]:
# --- Repo + SemanticRT data (auto-detect the flat layout + official txt split) ---
# Runs on Colab (Pro / A100) or Kaggle. The previous cell downloads the data and
# sets SEMRT_ROOT, which is checked first below; the other paths are fallbacks.
import os, sys, glob

REPO = os.getcwd()                 # current dir = the DiGToR repo
assert os.path.isfile(os.path.join(REPO, 'digtor', '__init__.py')), \
    f'Run this from the DiGToR repo root (no digtor/ package found in {REPO}).'

_mods, _splits = ['rgb', 'thermal', 'labels'], ['train.txt', 'val.txt', 'test.txt']

def _is_semrt_root(path):
    return (all(os.path.isdir(os.path.join(path, m)) for m in _mods)
            and all(os.path.isfile(os.path.join(path, s)) for s in _splits))

# Priority: SEMRT_ROOT (set by the download cell) -> repo/SemanticRT_dataset ->
# Colab /content or Drive -> Kaggle inputs.
_candidates = []
if os.environ.get('SEMRT_ROOT'):
    _candidates.append(os.environ['SEMRT_ROOT'])
_candidates.append(os.path.join(REPO, 'SemanticRT_dataset'))
_candidates += ['/content/SemanticRT_dataset', '/content/drive/MyDrive/SemanticRT_dataset']
_candidates += glob.glob('/kaggle/input/**/SemanticRT_dataset', recursive=True)
_candidates += glob.glob('/kaggle/input/*', recursive=False)

ROOT = None
for _cand in dict.fromkeys(_candidates):
    if _is_semrt_root(_cand):
        ROOT = _cand; break
assert ROOT is not None, 'SemanticRT data not found: need flat rgb/thermal/labels + train/val/test.txt.'

print('REPO =', REPO)
print('ROOT =', ROOT)
sys.path.insert(0, REPO)
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')


In [ ]:
# --- Hyper-parameters: hard-seg-loss forced paths (mirrors the FMB Phase-0 rig) ---
# 3-path DiGToR router (V-trust / T-rescue / Joint), rel_gate OFF at eval, hard
# seg_loss on force_path='v'/'t' plus teacher KD (--lambda_distill 0.5).
# The digtor.dataset.semanticrt loader uses the official split: train.txt/val.txt for
# train+val, test.txt as the held-out evaluation set.
import os
from digtor.models import build_model

# Guardrail: if this fails, the clone is not on the expected 3-path code.
_probe = build_model('digtor', base=8)
assert getattr(_probe.router[-1], 'out_channels', None) == 3, 'Expected the 3-path DiGToR router.'
del _probe

H, W, BASE = 384, 512, 32
BS = 8
EPOCHS = 80
DIGTOR_EPOCHS = 80
CKPT, RES = 'ckpt_semrt', 'results_semrt'
AMP = '--amp'                      # set '' to disable mixed precision
LR = '--lr 5e-4'
IGNORE_BG = '--ignore_bg'          # ignore class 0 in mIoU; set '' to include it
CORRUPT = '--corrupt_aug --corrupt_p 0.5'
DISTILL = '--lambda_distill 0.5'   # hard seg-loss forced paths + KD when teachers exist
NOGATE = '--disable_gate'          # train/eval with reliability gate OFF (rel_gate=0)
GAMMA_PRIOR = '--gamma_prior 2.0'
LAMBDA_COST = '--lambda_cost 0.1'
ROUTE_BETA = '--route_beta 0.7'

# DRY RUN: set LIMIT='8' to smoke-test all cells quickly; set '' for full train/test.
LIMIT = ''
LIMIT_ARG = f'--limit {LIMIT}' if LIMIT else ''
if LIMIT:
    os.environ['SEMRT_LIMIT'] = LIMIT
    EPOCHS = DIGTOR_EPOCHS = 1
else:
    os.environ.pop('SEMRT_LIMIT', None)

os.makedirs(CKPT, exist_ok=True); os.makedirs(RES, exist_ok=True)
print('config:', dict(H=H, W=W, BASE=BASE, BS=BS, EPOCHS=EPOCHS,
                      DIGTOR_EPOCHS=DIGTOR_EPOCHS, CKPT=CKPT, RES=RES,
                      LIMIT=LIMIT or 'full', lr=LR, ignore_bg=bool(IGNORE_BG),
                      corrupt_aug=CORRUPT, distill=DISTILL, gate='off',
                      route_beta=ROUTE_BETA))


In [ ]:
# --- A100 speed knobs (quality-neutral) ---
# channels_last + persistent dataloader workers are always on in the code. Here
# we add torch.compile (fuses the conv graph -> faster GPU step) and match the
# worker count to the host CPU cores so the GPU is never starved waiting on JPEG
# decode. None of this changes the maths, so results are preserved.
import os as _os
WORKERS = f"--workers {min(8, (_os.cpu_count() or 4))}"
COMPILE = '--compile'        # set '' to skip torch.compile (e.g. if it errors)
SPEED = f'{COMPILE} {WORKERS}'.strip()
#
# OPTIONAL, NOT quality-neutral: a bigger batch fills the A100 better but changes
# the optimisation. If you raise BS, scale LR by the same factor (linear scaling
# rule) to keep accuracy, e.g. BS=16 -> LR='--lr 1e-3'. Left at the pinned BS=8.
# BS = 16; LR = '--lr 1e-3'
print('SPEED =', repr(SPEED))


In [ ]:
# --- Weights & Biases: checkpoint sync (survive Colab disconnects) ---
# Logs metrics and uploads each best checkpoint as a wandb artifact (<mode>-ckpt).
# A dropped session can pull them back (next cell) so finished models are reused
# instead of retrained. Set USE_WANDB=False to disable everything.
import os, subprocess, sys

USE_WANDB = True
WB_PROJECT = 'digtor-semanticrt'
WB_ENTITY = None            # None = your default wandb entity

if USE_WANDB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'wandb'], check=True)
    import wandb
    # Kaggle: add your key as a secret named WANDB_API_KEY. Colab: this prompts
    # once, or set os.environ['WANDB_API_KEY'] = '...' before running.
    try:
        wandb.login()
    except Exception as e:
        print('wandb.login failed -> disabling wandb:', e); USE_WANDB = False

WANDB = (f'--wandb --wandb_project {WB_PROJECT}'
         + (f' --wandb_entity {WB_ENTITY}' if WB_ENTITY else '')) if USE_WANDB else ''
ENTITY_ARG = f'--entity {WB_ENTITY}' if WB_ENTITY else ''
print('wandb:', 'ON' if USE_WANDB else 'OFF', '| WANDB =', repr(WANDB))


In [ ]:
# --- Pull any already-trained checkpoints from wandb into CKPT ---
# Idempotent + failsafe: missing artifacts are skipped. After this, each training
# cell's `[ -f {CKPT}/x.pt ]` guard reuses whatever was recovered instead of
# retraining. Run this first on a fresh runtime to resume a dropped session.
if USE_WANDB:
    !python -m digtor.wandb_ckpt --dataset semanticrt --pull --project {WB_PROJECT} {ENTITY_ARG} --out {CKPT} --modes v_only t_only fusion digtor
else:
    print('wandb OFF -> skipping checkpoint pull')


In [ ]:
# Step 1 - V-only teacher (clean reference, train/val from train.txt/val.txt)
![ -f {CKPT}/v_only.pt ] && echo "[skip] {CKPT}/v_only.pt exists - reusing" || python -m digtor.train --dataset semanticrt --root {ROOT} --mode v_only --epochs {EPOCHS} --bs {BS} --height {H} --width {W} --base {BASE} --out {CKPT} {AMP} {LR} {IGNORE_BG} {WANDB} {SPEED}


In [ ]:
# Step 2 - T-only teacher (clean reference, train/val from train.txt/val.txt)
![ -f {CKPT}/t_only.pt ] && echo "[skip] {CKPT}/t_only.pt exists - reusing" || python -m digtor.train --dataset semanticrt --root {ROOT} --mode t_only --epochs {EPOCHS} --bs {BS} --height {H} --width {W} --base {BASE} --out {CKPT} {AMP} {LR} {IGNORE_BG} {WANDB} {SPEED}


In [ ]:
# Step 3 - rescue detector (decision gate) on the held-out test split
!python -m digtor.eval_detector --dataset semanticrt --root {ROOT} --height {H} --width {W} --base {BASE} --v_ckpt {CKPT}/v_only.pt --t_ckpt {CKPT}/t_only.pt --split test --out {RES}/detector.json


In [ ]:
# Step 4 - Fusion baseline (same corruption augmentation for a fair MFR comparison)
![ -f {CKPT}/fusion.pt ] && echo "[skip] {CKPT}/fusion.pt exists - reusing" || python -m digtor.train --dataset semanticrt --root {ROOT} --mode fusion --epochs {EPOCHS} --bs {BS} --height {H} --width {W} --base {BASE} --out {CKPT} {AMP} {LR} {IGNORE_BG} {WANDB} {SPEED} {CORRUPT}


### Step 5 - DiGToR

3-path DiGToR router (V-trust, T-rescue, Joint). The forced pure paths are
trained with hard segmentation loss (`force_path='v'/'t'`) plus teacher KD, so
catastrophic single-modality failure can cut to a strong fallback path.


In [ ]:
# Step 5 - DiGToR train with hard-seg-loss forced paths
![ -f {CKPT}/v_only.pt ] && [ -f {CKPT}/t_only.pt ] || echo 'WARN: missing teachers -> run Step 1/2 first'
![ -f {CKPT}/digtor.pt ] && echo "[skip] {CKPT}/digtor.pt exists - reusing" || python -m digtor.train --dataset semanticrt --root {ROOT} --mode digtor --epochs {DIGTOR_EPOCHS} --bs {BS} --height {H} --width {W} --base {BASE} --out {CKPT} --v_ckpt {CKPT}/v_only.pt --t_ckpt {CKPT}/t_only.pt {AMP} {LR} {IGNORE_BG} {WANDB} {SPEED} {CORRUPT} {GAMMA_PRIOR} {LAMBDA_COST} {ROUTE_BETA} {DISTILL} {NOGATE}

# Clean mIoU + rescue protocol + routing analyses on the held-out test split. rel_gate=0 at eval.
!python -m digtor.eval_rescue --dataset semanticrt --root {ROOT} --height {H} --width {W} --base {BASE} --ckpt_dir {CKPT} --rel_gate 0 --out {RES}/eval_rescue.json {IGNORE_BG}


In [ ]:
# Step 6 - corruption robustness, FLOPs, and the modality-cut win check
# Dense-routing robustness table (no retrain).
!python -m digtor.eval_robustness --dataset semanticrt --root {ROOT} --height {H} --width {W} --base {BASE} --ckpt_dir {CKPT} --rel_gate 0 --out {RES}/robustness.json {IGNORE_BG} {LIMIT_ARG}

# Route-share / FLOP accounting for the 3-path model.
!python -m digtor.profile_flops --dataset semanticrt --root {ROOT} --ckpt {CKPT}/digtor.pt --base {BASE} --height {H} --width {W} --out {RES}/flops.json {LIMIT_ARG}

# Catastrophic-thermal check with realizable force_path cuts.
!python -m digtor.eval_modality_cut --dataset semanticrt --root {ROOT} --height {H} --width {W} --base {BASE} --ckpt_dir {CKPT} --skips 0.3 0.5 0.6 --out {RES}/modality_cut.json {IGNORE_BG} {LIMIT_ARG}


### Step 7 - Optional: per-condition test sub-splits

SemanticRT ships test sub-splits (`test_day`, `test_night`, `test_mc`,
`test_mo`, `test_hard`). The loader exposes them via `test_split=...`; below is a
quick day-vs-night mIoU read on the DiGToR model (skip if you only need the main
table).


In [ ]:
import json
import torch
from digtor import IGNORE_INDEX, get_dataset_config
NUM_CLASSES = get_dataset_config('semanticrt').num_classes
from digtor.dataset.semanticrt import build_loaders
from digtor.models import build_model
from digtor.metrics import confusion_matrix, metrics_from_cm
import numpy as np

device = 'cuda' if torch.cuda.is_available() else 'cpu'
ig = (0,) if IGNORE_BG else ()
ck = torch.load(f'{CKPT}/digtor.pt', map_location=device)
model = build_model('digtor', base=BASE, num_classes=NUM_CLASSES).to(device).eval()
model.load_state_dict(ck['model'])

sub = {}
for ts in ['test_day', 'test_night', 'test_hard']:
    _, _, el = build_loaders(ROOT, size=(H, W), batch_size=1, num_workers=2, test_split=ts)
    cm = np.zeros((NUM_CLASSES, NUM_CLASSES), np.int64)
    with torch.no_grad():
        for b in el:
            logit = model(b['rgb'].to(device), b['ir'].to(device), rel_gate=0)
            cm += confusion_matrix(logit.argmax(1).cpu().numpy(), b['label'].numpy(), NUM_CLASSES, IGNORE_INDEX)
    m = metrics_from_cm(cm, ignore_classes=ig)
    sub[ts] = m['mIoU']
    print(f'{ts:12s} n={len(el.dataset):5d}  mIoU={m["mIoU"]:.4f}')
json.dump(sub, open(f'{RES}/subsplit_miou.json', 'w'), indent=2)


In [ ]:
# Step 8 - result summary
import json, os

def _ld(path):
    try:
        return json.load(open(path))
    except Exception as e:
        print(f'[missing] {path}: {e}'); return None

er = _ld(f'{RES}/eval_rescue.json'); rob = _ld(f'{RES}/robustness.json')
flp = _ld(f'{RES}/flops.json'); cut = _ld(f'{RES}/modality_cut.json')

def row(name, evidence, verdict=''):
    print(f'{name:30s} | {evidence:70s} | {verdict}')

print('=' * 128)
print('DiGToR on SemanticRT: hard-seg-loss forced paths, official split')
print('=' * 128)
if er and 'digtor' in er and 'fusion' in er:
    dm, fm = er['digtor']['mIoU'], er['fusion']['mIoU']
    row('clean mIoU', f'digtor {dm:.4f} vs fusion {fm:.4f} (delta {dm-fm:+.4f})',
        'WIN' if dm >= fm else 'below fusion')
if er and er.get('digtor_rescue') and er.get('fusion_rescue'):
    dr, fr = er['digtor_rescue'], er['fusion_rescue']
    row('rescue protocol',
        f"TRR {dr['TRR']:.3f}/{fr['TRR']:.3f}, VPR {dr['VPR']:.3f}/{fr['VPR']:.3f}, HRR {dr['HRR']:.3f}/{fr['HRR']:.3f}",
        'compare vs fusion')
if rob and rob.get('MFR'):
    m = rob['MFR']
    d = m.get('digtor', {}).get('thermal_MFR', float('nan'))
    f = m.get('fusion', {}).get('thermal_MFR', float('nan'))
    row('thermal MFR', f'digtor {d:.3f} vs fusion {f:.3f}', 'WIN' if d >= f else 'below fusion')
if cut and cut.get('verdict'):
    row('modality-cut', str(cut['verdict'])[:70])
if flp and flp.get('flops_gflops'):
    fg = flp['flops_gflops']
    if 'digtor' in fg and 'fusion' in fg:
        row('FLOPs', f"digtor {fg['digtor']:.1f} vs fusion {fg['fusion']:.1f} GFLOPs ({fg['digtor']/fg['fusion']:.2f}x)")
print('=' * 128)


In [ ]:
# Collect all JSON results for download (Output tab)
import json, glob
for f in sorted(glob.glob(f'{RES}/*.json')):
    print('=' * 60, '\n', f)
    print(json.dumps(json.load(open(f)), indent=2)[:1500])
